In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
%pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 5.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 9.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 12.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━

In [3]:
import mlflow
import dagshub

In [4]:
dagshub.init(repo_owner='mesata', repo_name='IEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f3707fa8-dc26-4894-8764-7fb3cc08145e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b8378f248d0888343aaff3664a55403357b6c316e3119e7aec9e41362830b480




Accessing as mesata

Initialized MLflow to track repo "mesata/IEE-CIS-Fraud-Detection"

Repository mesata/IEE-CIS-Fraud-Detection initialized!

In [5]:
model_name = "base_model_xgboost"
model_version =1
model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}/{model_version}")

In [6]:
test_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
test = pd.merge(test_trans, test_id, on='TransactionID', how='left')

In [7]:
test.columns = [col.replace('_', '-') if col.startswith('id') else col for col in test.columns]
test['TransactionHour'] = (test['TransactionDT'] // 3600) % 24

In [10]:
from sklearn.preprocessing import LabelEncoder
cat_cols = test.select_dtypes(include=['object']).columns
for col in cat_cols:
    le = LabelEncoder()
    test[col] = le.fit_transform(test[col].astype(str))

In [15]:
train_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
train = pd.merge(train_trans, train_id, on='TransactionID', how='left')

In [16]:
train['TransactionHour'] = (train['TransactionDT'] // 3600) % 24
for col in train.select_dtypes(include=['object']).columns:
    train[col] = LabelEncoder().fit_transform(train[col].astype(str))

In [17]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
num_features = train.select_dtypes(include=[np.number]).columns.drop(['isFraud', 'TransactionID', 'TransactionDT'])
selector.fit(train[num_features])
features_to_keep = num_features[selector.get_support()]

all_selected = list(features_to_keep) + list(train.select_dtypes(include=['object']).columns)
train_subset = train[all_selected]

In [21]:
corr_matrix = train_subset.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
final_features = [c for c in train_subset.columns if c not in to_drop]
print(f"Final Features Selected: {len(final_features)}")

Final Features Selected: 268


In [27]:
existing_features = [c for c in final_features if c in test.columns]
X_test = test[existing_features]

In [31]:
model_xgb = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=8, 
    learning_rate=0.05, 
    tree_method='hist', 
    random_state=42,
    n_jobs=-1
)

In [32]:
model_xgb.fit(X_train, y_train)

preds = model_xgb.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({'TransactionID': test['TransactionID'], 'isFraud': preds})
submission.to_csv('submission.csv', index=False)
print("წარმატებით დასრულდა")

წარმატებით დასრულდა
